🧱 Bloque 1 – Importación de librerías

Este bloque reúne todas las herramientas necesarias para el proceso completo de automatización:
- Manejo de datos con pandas y numpy.
- Conexión a SQL Server mediante sqlalchemy.
- Manipulación de rutas, archivos y fechas con Path, shutil, datetime y time.
- Limpieza y normalización de textos con re y unicodedata.

Con estas librerías ya está preparado el entorno para leer, transformar y cargar los datos en SQL Server.

In [ ]:
import pandas as pd
import re
import unicodedata
import numpy as np
from datetime import datetime
from sqlalchemy import text
import sys
import importlib
import json
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/concreces/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path.cwd().resolve().parents[2]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()
CONFIG_DIR    = (PROJECT_ROOT / "config").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:",    CONFIG_DIR,    "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"
try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)
    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))
except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "concreces", "config_concreces.json")
try:
    with open(config_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()))
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")


🔌 Bloque 2 – Configuración de la conexión a SQL Server

Este bloque establece la conexión hacia el servidor y la base de datos donde se insertarán los datos:
- Se indica el servidor, base de datos, esquema y tabla de destino.
- Se construye la cadena de conexión (connection_string) con autenticación integrada.
- Se crea un engine optimizado con fast_executemany para cargas masivas eficientes y pool_pre_ping para evitar desconexiones.

Aquí queda lista la conexión para permitir enviar datos a SQL sin interrupciones.

In [ ]:
# --- Parámetros de conexión ---
server   = data["server_config"]["server"]
database = data["server_config"]["database"]
schema   = data["server_config"]["schema"]
table    = data["tablas_remotas"]["tabla_principal"]

driver = "ODBC Driver 17 for SQL Server"

# ODBC connection string (pyodbc) → para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise"
)


📂 Bloque 3 – Definición de la carpeta de trabajo y archivos permitidos

Este bloque establece el origen de los archivos que se procesarán:
- RUTA_ARCHIVOS apunta a la carpeta donde se encuentran los Excel.
- EXTS indica qué extensiones se permiten (solo .xlsx).
- PREFERRED_SHEETS lista los nombres de hojas que se espera encontrar.

Con esta información, el script sabe dónde buscar y qué formatos aceptar.

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "concreces").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

EXTS = (".xlsx", "")  # Extensiones de archivo permitidas

PREFERRED_SHEETS = ["Base", "base"]
FALLBACK_SHEET_INDEX = 0


🧹 Bloque 4 – Normalización de textos y nombres de columnas

Este bloque prepara funciones auxiliares para limpiar textos:
- _strip_accents elimina tildes y caracteres especiales.
- normalize_name transforma nombres de columnas a un formato uniforme:
    - minúsculas,
    - sin espacios extras,
    - con _ como separador,
    - sin símbolos raros.

Estas funciones permiten trabajar con encabezados consistentes sin importar cómo vengan en el Excel original.

🔎 Bloque 5 – Selección del archivo más reciente y hoja de trabajo

Este bloque automáticamente identifica el archivo que se debe procesar:
    Verifica que la carpeta exista.
    Lista los archivos válidos y los ordena por fecha de modificación.
    Selecciona el archivo más reciente.
    Intenta abrirlo; si está bloqueado (por ejemplo por OneDrive), crea una copia temporal para poder leerlo.
    Detecta la hoja correcta dentro del Excel entre las preferidas.

Aquí se asegura que siempre se procese el archivo actualizado y la hoja correspondiente.

In [ ]:
info = fun.get_latest_excel_and_sheet(
    RUTA_ARCHIVOS,
    EXTS,
    PREFERRED_SHEETS,
    fallback_sheet_index=FALLBACK_SHEET_INDEX,
    recursive=False,
    engine="openpyxl",
)

archivo_origen  = info["archivo_origen"]
excel_path_used = info["excel_path_used"]
_tmp_copy_path  = info["tmp_copy_path"]
target          = info["target_sheet"]
sheet_names     = info["sheet_names"]

print(f"Archivo: {archivo_origen.name}")
print(f"Hoja seleccionada: {target}")


📑 Bloque 6 – Lectura segura del Excel y normalización del encabezado

Este bloque carga la hoja seleccionada de forma segura:
- Usa read_excel_safe para evitar errores inesperados.
- Limpia valores como "nan" o "None".
- Normaliza todos los encabezados con las funciones del Bloque 4.
- Muestra cuántas filas y columnas tiene el archivo y enseña los encabezados resultantes.

Después de este paso, el DataFrame queda limpio y estandarizado, listo para mapear columnas.

In [7]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = fun.read_excel_safe(io, target)

if _tmp_copy_path is not None and _tmp_copy_path.exists():
    try:
        _tmp_copy_path.unlink()
        print(f"🗑️ Copia temporal eliminada: {_tmp_copy_path}")
    except Exception as e:
        print(f"⚠️ No se pudo borrar la copia temporal '{_tmp_copy_path}': {e}")

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "nan": "", "NaN": ""})
df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]
print("Encabezados normalizados:", list(df_raw.columns))
df_raw.head()

Encabezados normalizados: ['nro_operacion', 'poliza', 'serie_r_u_t', 'digito_r_u_t', 'apellido_paterno', 'apellido_materno', 'nombres', 'fecha_nacimiento', 'fecha_escritura', 'plazo_no_anos', 'factor', 'factor_mensual', 'prima_uf', 'prima_pesos', 'fecha_conversion', 'capital', 'credito', 'saldo_insoluto', 'cia_aseguradora']


,nro_operacion,poliza,serie_r_u_t,digito_r_u_t,apellido_paterno,apellido_materno,nombres,fecha_nacimiento,fecha_escritura,plazo_no_anos,factor,factor_mensual,prima_uf,prima_pesos,fecha_conversion,capital,credito,saldo_insoluto,cia_aseguradora
0,01-01-00573-9,8598971,13173687,8,PALMA,FLORES,CARLA ANDREA,1974-12-03 00:00:00,2001-01-25 00:00:00,25,24,0.0401,0.2647,10492,2025-11-30 00:00:00,660,660,9.08492,Seguros Generales Suramericana S.A.
1,01-01-00586-K,8598971,09781171,7,ROJAS,GONZALEZ,FRANCISCO JAVIER,1966-03-14 00:00:00,2000-12-29 00:00:00,25,24,0.0401,0.2819,11176,2025-11-30 00:00:00,703,703,12.02482,Seguros Generales Suramericana S.A.
2,01-01-00621-2,8598971,09270618,4,CONTRERAS,MANDUCHER,ANA ROSA,1966-08-30 00:00:00,2001-01-11 00:00:00,25,24,0.0401,0.1925,7631,2025-11-30 00:00:00,480,480,12.18192,Seguros Generales Suramericana S.A.
3,01-01-00629-8,8598971,08939557,7,SAAVEDRA,FUNES,HILDA LEONTINA,1964-04-12 00:00:00,2001-01-25 00:00:00,25,24,0.0401,0.2727,10810,2025-11-30 00:00:00,680,680,17.25772,Seguros Generales Suramericana S.A.
4,01-01-00630-1,8598971,12571051,4,GONZALEZ,GONZALEZ,VERONICA ISOLINA,1974-10-08 00:00:00,2001-01-25 00:00:00,25,24,0.0401,0.1692,6709,2025-11-30 00:00:00,422,422,10.93782,Seguros Generales Suramericana S.A.


🧬 Bloque 7 – Mapeo de columnas del Excel hacia columnas estándar

Este bloque construye el DataFrame estructurado que se usará para cargar en SQL:
- La función pick busca la primera columna válida entre distintas variantes (por ejemplo "nro_operacion", "operacion", "folio", etc.).
- Para cada campo (RUT, DV, Capital, etc.) se toma la columna correcta sin importar cómo se llame en el Excel.
- Con esas columnas se forma el DataFrame df_can con nombres estándar como "Guia", "Poliza", "Saldo_Insoluto", etc.
- Se agrega también el nombre del archivo procesado.

Este bloque actúa como un traductor entre diferentes estructuras de Excel y tu estructura final de SQL.

In [8]:
def pick(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series([None]*len(df), index=df.index)


# Origen → Destino
guia                = pick(df_raw, "guia")
operacion           = pick(df_raw, "nro_operacion","n_operacion", "no_operacion", "noperacion", "operacion", "folio", "foliocredito")
poliza              = pick(df_raw, "poliza", "póliza", "n_poliza", "npoliza")
rut                 = pick(df_raw, "serie_r_u_t", "rut_afiliado", "rutafiliado", "rut", "rutfiliado")
dv                  = pick(df_raw, "digito_r_u_t", "dv_afiliado", "dvafiliado", "dv", "dvfiliado")
fecha_nacimiento    = pick(df_raw, "fecha_nacimiento", "fechanacimiento", "fecha_de_nacimiento", "fechadenacimiento")
fecha_escritura     = pick(df_raw, "fecha_escritura", "fechaescritura", "fecha_de_escritura", "fechadeescritura")
plazos_anuales      = pick(df_raw, "plazo_no_anos", "plazosanuales", "plazos_anual", "plazosanual",)
factor              = pick(df_raw, "factor")
factor_mensual      = pick(df_raw, "factor_mensual", "factormensual")
prima_uf            = pick(df_raw, "prima_uf", "primauf")
prima_pesos         = pick(df_raw, "prima_pesos", "primapesos")
fecha_conversion    = pick(df_raw, "fecha_conversion", "fechaconversion", "fecha_de_conversion", "fechadeconversion")
capital             = pick(df_raw, "capital_asegurado", "capitalasegurado", "capital")
credito             = pick(df_raw, "credito_asegurado", "creditoasegurado", "credito")
saldo_insoluto      = pick(df_raw, "saldo_insoluto", "saldoinsoluto")


df_can = pd.DataFrame({
    "Guia": guia,
    "Operacion": operacion,
    "Poliza": poliza,
    "RUT": rut,
    "DV": dv,
    "Fecha_Nacimiento": fecha_nacimiento,
    "Fecha_Escritura": fecha_escritura,
    "Plazos_Anuales": plazos_anuales,
    "Factor": factor,
    "FactorMensual": factor_mensual,
    "Prima_UF": prima_uf,
    "Prima_Pesos": prima_pesos,
    "Fecha_Conversion": fecha_conversion,
    "Capital": capital,
    "Credito": credito,
    "Saldo_Insoluto": saldo_insoluto,
    "NOMBRE_ARCHIVO": archivo_origen.name,

})

df_can.head()

,Guia,Operacion,Poliza,RUT,DV,Fecha_Nacimiento,Fecha_Escritura,Plazos_Anuales,Factor,FactorMensual,Prima_UF,Prima_Pesos,Fecha_Conversion,Capital,Credito,Saldo_Insoluto,NOMBRE_ARCHIVO
0,None,01-01-00573-9,8598971,13173687,8,1974-12-03 00:00:00,2001-01-25 00:00:00,25,24,0.0401,0.2647,10492,2025-11-30 00:00:00,660,660,9.08492,CONCRECES Nomina Cesantia Noviembre 2025.xlsx
1,None,01-01-00586-K,8598971,09781171,7,1966-03-14 00:00:00,2000-12-29 00:00:00,25,24,0.0401,0.2819,11176,2025-11-30 00:00:00,703,703,12.02482,CONCRECES Nomina Cesantia Noviembre 2025.xlsx
2,None,01-01-00621-2,8598971,09270618,4,1966-08-30 00:00:00,2001-01-11 00:00:00,25,24,0.0401,0.1925,7631,2025-11-30 00:00:00,480,480,12.18192,CONCRECES Nomina Cesantia Noviembre 2025.xlsx
3,None,01-01-00629-8,8598971,08939557,7,1964-04-12 00:00:00,2001-01-25 00:00:00,25,24,0.0401,0.2727,10810,2025-11-30 00:00:00,680,680,17.25772,CONCRECES Nomina Cesantia Noviembre 2025.xlsx
4,None,01-01-00630-1,8598971,12571051,4,1974-10-08 00:00:00,2001-01-25 00:00:00,25,24,0.0401,0.1692,6709,2025-11-30 00:00:00,422,422,10.93782,CONCRECES Nomina Cesantia Noviembre 2025.xlsx


🧮 Bloque 8 – Asignación de Plan Técnico según la Póliza

Este bloque incorpora la lógica de negocio:
- Se crea un diccionario que relaciona cada póliza con su respectivo plan técnico.
- Se agrega la columna "Plan_Tecnico" aplicando ese mapeo.
- Se imprime la cantidad de filas por cada plan.
- Se detectan pólizas que no tienen un plan asignado para revisarlas manualmente.

Así se garantiza que cada póliza reciba el plan técnico correspondiente.

In [9]:
 # Diccionario de mapeo Póliza → Plan_Tecnico
MAPA_PLANES = {
    "8598974":	4783,
    "8598978":	8621,
    "8598975":	8618,
    "8598976":	8619,
    "8598977":	8620,
    "8598971":	4784,
    "8598973":	5677,
    "8598972":	4780,
}

df_can["Plan_Tecnico"] = df_can["Poliza"].map(MAPA_PLANES)

print("📊 Cantidad de filas por Plan_Tecnico:")
print(df_can["Plan_Tecnico"].value_counts(dropna=False))

faltantes = df_can[df_can["Plan_Tecnico"].isna()]["Poliza"].unique()
if len(faltantes) > 0:
    print("⚠️ Pólizas sin Plan_Tecnico definido en el mapa:", faltantes)

📊 Cantidad de filas por Plan_Tecnico:
Plan_Tecnico
4783    3032
4780    1059
8618     604
5677     173
8620     161
4784     109
8619       9
8621       2
Name: count, dtype: int64


🗂️ Bloque 9 – Generación del nombre de archivo canónico

Este bloque estandariza la forma en que se registra el archivo:
- Detecta el período YYYYMM en el nombre del archivo, sin importar su formato.
- Crea un nombre canónico del tipo "YYYYMM CONCRECES Nominas cesantia.xlsx".
- Si no se logra detectar el período, genera un nombre alternativo con un timestamp.
- Finalmente actualiza la columna "NOMBRE_ARCHIVO" con el nombre estandarizado.

Esto permite mantener consistencia y evitar duplicados en la base de datos.

In [10]:
def _strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))

MESES = {
    "ENERO": "01", "ENE": "01",
    "FEBRERO": "02", "FEB": "02",
    "MARZO": "03", "MAR": "03",
    "ABRIL": "04", "ABR": "04",
    "MAYO": "05", "MAY": "05",
    "JUNIO": "06", "JUN": "06",
    "JULIO": "07", "JUL": "07",
    "AGOSTO": "08", "AGO": "08",
    "SEPTIEMBRE": "09", "SETIEMBRE": "09", "SEP": "09", "SET": "09",
    "OCTUBRE": "10", "OCT": "10",
    "NOVIEMBRE": "11", "NOV": "11",
    "DICIEMBRE": "12", "DIC": "12",
}


def _extract_yyyymm_from_name(nombre: str) -> str:
    """
    Intenta extraer YYYYMM desde el nombre del archivo (sin importar la extensión).
    Cubre:
      - YYYYMM
      - YYYY[-_/ .]?MM
      - MM[-_/ .]?YYYY
      - MES(ES) + YYYY (con tildes/abreviaturas) en cualquier orden
    Lanza ValueError si no encuentra período.
    """
    stem = Path(nombre).stem
    stem_norm = _strip_accents(stem).upper()

    m = re.search(r"(20\d{2})(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(20\d{2})[-_/.\s]?(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(0[1-9]|1[0-2])[-_/.\s]?(20\d{2})", stem_norm)
    if m:
        return f"{m.group(2)}{m.group(1)}"

    m_year = re.search(r"(20\d{2})", stem_norm)
    if m_year:
        year = m_year.group(1)
        for mes_txt, mm in MESES.items():
            if re.search(rf"(?<![A-Z0-9]){mes_txt}(?![A-Z0-9])", stem_norm):
                return f"{year}{mm}"

    raise ValueError(f"No pude extraer YYYYMM desde el nombre: {nombre}")


def canonicalizar_planes(nombre: str) -> str:
    """
    Devuelve un nombre canónico estandarizado: 'SC_MARSHYYYYMM.xlsx'
    Si no logra extraer el período (YYYYMM), devuelve un nombre genérico con aviso.
    """
    try:
        yyyymm = _extract_yyyymm_from_name(nombre)
        return f"{yyyymm} CONCRECES Nominas cesantia.xlsx"
    except ValueError as e:
        print(f"⚠️ No se pudo extraer período desde el nombre '{nombre}'. "
              f"Se usará un nombre genérico. Detalle: {e}")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"{timestamp} CONCRECES Nominas cesantia.xlsx"


nombre_canonico = canonicalizar_planes(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)


df_can["NOMBRE_ARCHIVO"] = nombre_canonico

Nombre original : CONCRECES Nomina Cesantia Noviembre 2025.xlsx
Nombre canónico : 202511 CONCRECES Nominas cesantia.xlsx


🔢 Bloque 10 – Limpieza de la columna Operacion y creación de Operacion2

Este bloque extrae la parte numérica de la operación:
- _only_digits_series elimina letras, guiones y caracteres especiales.
- Si la columna "Operacion" no existe, se crea vacía para evitar errores.
- "Operacion2" contiene solo los dígitos (por ejemplo "BCH00010406" → 10406).
- Se muestra cuántas filas fueron modificadas y algunos ejemplos.

Esto deja la operación en un formato ideal para consultas numéricas y almacenamiento como bigint.

In [11]:
def _only_digits_series(s: pd.Series) -> pd.Series:
    out = s.astype(str).str.replace(r"\D+", "", regex=True)
    out = out.replace("", np.nan)
    return out

if "Operacion" not in df_can.columns:
    df_can["Operacion"] = np.nan

df_can["Operacion2"]    = _only_digits_series(df_can["Operacion"])

m_op  = df_can["Operacion"].astype(str)   != df_can["Operacion2"].astype(str)

print("=== RESUMEN LIMPIEZA NUMÉRICA ===")
print(f"Filas afectadas en Operacion  -> {int(m_op.sum()):,}")

ej_op  = df_can.loc[m_op,  ["Operacion", "Operacion2"]].head(5)

if not ej_op.empty:
    print("\nEjemplos Operacion → Operacion2:")
    display(ej_op)

=== RESUMEN LIMPIEZA NUMÉRICA ===
Filas afectadas en Operacion  -> 5,149

Ejemplos Operacion → Operacion2:


,Operacion,Operacion2
0,01-01-00573-9,0101005739
1,01-01-00586-K,010100586
2,01-01-00621-2,0101006212
3,01-01-00629-8,0101006298
4,01-01-00630-1,0101006301


📆 Bloque 11 – Generación del período MES_RECAUDACION

Este bloque extrae el período desde el nombre del archivo:
- Usa una expresión regular para identificar el YYYYMM.
- Crea la columna "MES_RECAUDACION" con ese valor.
- Si algún período no puede detectarse, se avisa.
- Muestra ejemplos para verificar visualmente.

Aquí queda definida la columna que permite clasificar o filtrar los datos por período.

In [12]:
def extraer_yyyymm_de_nombre_archivo(df: pd.DataFrame, nombre_columna: str = "NOMBRE_ARCHIVO") -> pd.Series:
    return df[nombre_columna].str.extract(r"(20\d{2})(0[1-9]|1[0-2])", expand=False).apply(lambda x: ''.join(x), axis=1)

df_can["MES_RECAUDACION"] = extraer_yyyymm_de_nombre_archivo(df_can)

if df_can["MES_RECAUDACION"].isnull().any():
    print("⚠️ Algunos valores de MES_RECAUDACION no pudieron ser extraídos.")
    
df_can[["NOMBRE_ARCHIVO", "MES_RECAUDACION"]].head(3)

,NOMBRE_ARCHIVO,MES_RECAUDACION
0,202511 CONCRECES Nominas cesantia.xlsx,202511
1,202511 CONCRECES Nominas cesantia.xlsx,202511
2,202511 CONCRECES Nominas cesantia.xlsx,202511


🏷️ Bloque 12 – Tipificación y preparación del DataFrame para SQL

Este bloque garantiza que cada columna tenga el tipo correcto antes de insertarse en SQL:
- Convierte textos vacíos a NaN.
- Pasa las fechas a tipo datetime.
- Transforma:
    - enteros → "Int64" (acepta nulls),
    - decimales → float64,
    - bigints → "Int64".
- Redondea valores cuando una columna debería ser entera pero llega con decimales.
- Limpia columnas de texto y limita su longitud.
- Muestra los tipos finales y revisa nulos en columnas importantes.
- Finalmente crea df_sql con el orden exacto de columnas que espera SQL.

Este bloque deja el DataFrame perfectamente alineado al modelo de la tabla en SQL Server.

In [13]:

INT_COLS    = ["Guia", "Poliza", "RUT", "DV", "Plazos_Anuales", "Prima_Pesos", "Capital", "Credito", "Plan_Tecnico", "MES_RECAUDACION"]
BIGINT_COLS = ["Operacion2"]
FLOAT_COLS  = ["Factor", "FactorMensual", "Prima_UF", "Saldo_Insoluto"]
STR_COLS    = ["Operacion", "NOMBRE_ARCHIVO"]
DATE_COLS   = ["Fecha_Nacimiento", "Fecha_Escritura", "Fecha_Conversion"]


def _to_num_series(s: pd.Series) -> pd.Series:
    s2 = s.astype(str).str.strip().replace({
        "": np.nan,
        "None": np.nan,
        "none": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
    })
    return pd.to_numeric(s2, errors="coerce")


def convert_to_datetime(df: pd.DataFrame, date_columns: list) -> pd.DataFrame:
    """
    Convierte las columnas de fechas de formato object a formato datetime (YYYY-MM-DD).
    Si la conversión falla, reemplaza con NaT (Not a Time).
    """
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False, infer_datetime_format=True)
    return df

df_can = convert_to_datetime(df_can, DATE_COLS)


# BIGINT -> pandas Int64 (acepta NaN)
for c in BIGINT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        s = _to_num_series(df_can[c])

        frac_mask = (s.notna()) & (s % 1 != 0)
        if frac_mask.any():
            print(f"⚠️ Columna '{c}' (INT) tiene {frac_mask.sum()} valores con decimales. Se redondearán antes de convertir a Int64.")
            # Aproximar al entero más cercano
            s = s.round(0)

        df_can[c] = s.astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("float64")



if "Operacion" in df_can.columns:
    df_can["Operacion"] = df_can["Operacion"].astype("string").str.strip().str.slice(0, 525)

if "NOMBRE_ARCHIVO" in df_can.columns:
    df_can["NOMBRE_ARCHIVO"] = df_can["NOMBRE_ARCHIVO"].astype("string").str.strip().str.slice(0, 255)

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)


criticas = ["Operacion","RUT","Poliza","NOMBRE_ARCHIVO"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")


cols_sql = [
    "Guia", "Operacion", "Poliza", "RUT", "DV", "Fecha_Nacimiento", "Fecha_Escritura", "Plazos_Anuales",
    "Factor", "FactorMensual", "Prima_UF", "Prima_Pesos", "Fecha_Conversion", "Capital",
    "Credito", "Saldo_Insoluto", "Plan_Tecnico", "NOMBRE_ARCHIVO", "MES_RECAUDACION", "Operacion2", 
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

⚠️ Columna 'Capital' (INT) tiene 733 valores con decimales. Se redondearán antes de convertir a Int64.
⚠️ Columna 'Credito' (INT) tiene 733 valores con decimales. Se redondearán antes de convertir a Int64.
✅ dtypes DESPUÉS:
 Guia                         Int64
Operacion           string[python]
Poliza                       Int64
RUT                          Int64
DV                           Int64
Fecha_Nacimiento    datetime64[ns]
Fecha_Escritura     datetime64[ns]
Plazos_Anuales               Int64
Factor                     float64
FactorMensual              float64
Prima_UF                   float64
Prima_Pesos                  Int64
Fecha_Conversion    datetime64[ns]
Capital                      Int64
Credito                      Int64
Saldo_Insoluto             float64
NOMBRE_ARCHIVO      string[python]
Plan_Tecnico                 Int64
Operacion2                   Int64
MES_RECAUDACION              Int64
dtype: object

🔎 Nulos en columnas críticas:
 - Operacion: 0 nulos
 - RUT: 

C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_20588\3607843725.py:26: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_20588\3607843725.py:26: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_20588\3607843725.py:26: UserWarning: The argument 'infer_datetime_format' is deprecat

🚦 Bloque 13 – Verificación de NOMBRE_ARCHIVO y Control de Cargas Previas

Este bloque garantiza que solo se carguen archivos válidos y que la carga sea segura:

- Verifica que la columna NOMBRE_ARCHIVO exista en el DataFrame.
- Normaliza sus valores (quita espacios, controla vacíos).
- Identifica todos los valores únicos (útil para consolidados con varios meses).
- Muestra un conteo por cada archivo detectado.

Prepara la variable vals, que será usada para decidir:

- Si cada archivo será insertado, o si será reemplazado en SQL Server.

In [14]:
assert "NOMBRE_ARCHIVO" in df_sql.columns, "Falta la columna 'NOMBRE_ARCHIVO' en df_sql."

df_sql["NOMBRE_ARCHIVO"] = (
    df_sql["NOMBRE_ARCHIVO"]
    .astype("string")
    .str.strip()
)


vals = [
    v for v in df_sql["NOMBRE_ARCHIVO"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'NOMBRE_ARCHIVO'.")

counts = (
    df_sql["NOMBRE_ARCHIVO"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} NOMBRE_ARCHIVO distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 NOMBRE_ARCHIVO distintos en el df_sql:
   - 202511 CONCRECES Nominas cesantia.xlsx: 5149 filas


🚀 Bloque 13 – Carga Dinámica a SQL Server + Reemplazo Inteligente

Este bloque realiza la carga final al Data Warehouse de forma controlada y segura.

Para cada NOMBRE_ARCHIVO detectado:

- Verifica si ya existe en SQL Server.    
- Si existe → elimina esas filas.   
- Inserta los datos nuevos del archivo (o consolidado).   
- Todo dentro de una sola transacción, evitando inconsistencias.    
- Genera un resumen detallado por archivo: reemplazo o inserción nueva.


In [15]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    for nombre_archivo in vals:  
        df_sub = df_sql[df_sql["NOMBRE_ARCHIVO"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para NOMBRE_ARCHIVO = '{nombre_archivo}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para NOMBRE_ARCHIVO = '{nombre_archivo}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para NOMBRE_ARCHIVO = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=table,
            con=conn,       
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para NOMBRE_ARCHIVO = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por NOMBRE_ARCHIVO:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")

🆕 No hay data previa para NOMBRE_ARCHIVO = '202511 CONCRECES Nominas cesantia.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 5149 filas para NOMBRE_ARCHIVO = '202511 CONCRECES Nominas cesantia.xlsx'.

📊 Resumen de carga por NOMBRE_ARCHIVO:
   - 202511 CONCRECES Nominas cesantia.xlsx: 5149 filas cargadas (archivo nuevo).


🗑️ Bloque 14 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [17]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\Concreces\CONCRECES Nomina Cesantia Octubre 2025.xlsx
